In [ ]:
import os

# Change this variable if the root folder name has been changed
root_dir = "nvae-shape-encoding"
current_dir = os.getcwd()

if not current_dir.endswith(root_dir):
    %cd ../..

assert os.getcwd().endswith(root_dir)

In [ ]:
import lightning as L
import torch

from arch.nvae.nvae import NVAE
from utils.const import SEED
from data_modules.acdc import ACDCMaskDataModule
from utils.utils import setup_device

model_path = "logs/nvae_acdc/best/default/checkpoints/epoch=97-step=20972.ckpt"

# Setup device
device = setup_device()
print(f"Device: {device}")

# Seed
L.seed_everything(SEED)

# Load data
data_module = ACDCMaskDataModule(batch_size=20)

# Reseed after preprocessing data
L.seed_everything(SEED)

# Load model
model = NVAE.load_from_checkpoint(model_path)

In [ ]:
from utils.utils import get_data

loader_test = data_module.test_dataloader()
feats = get_data(loader_test)
feats.shape

In [ ]:
num_data = feats.shape[0]

with torch.no_grad():
    model.eval()
    model.to(device)
    
    x_fake = model.decoder.generate(num_samples=num_data, device=device, temp=0.7)
    feats_fake = model.conditional_coder(x_fake)

In [ ]:
from utils.utils import show_samples

# Discretise probabilistic map then view generations
generations = torch.argmax(feats_fake[:24], dim=1).unsqueeze(1)
show_samples(generations, rgb=False, ncol=6, figsize=(24, 16))

In [ ]:
from utils.anatomical_validity_checker import AnatomicalValidityChecker
from utils.const import FRDS_MODEL_PATH
from utils.eval import compute_frds
from utils.utils import discretise

discretised_feats_fake = discretise(feats_fake)

# Percentage of anatomically valid generations
num_valid = 0

for discretised_feat_fake in discretised_feats_fake:
    AV = AnatomicalValidityChecker(discretised_feat_fake)
    if AV.count_violations() == 0:
        num_valid += 1

print(f"% Anatomically valid: {num_valid / num_data}")

# Compute FRDS
frds_value = compute_frds(
    feats,
    discretised_feats_fake,
    resnet_path=FRDS_MODEL_PATH,
    device=device,
)

print(f"FRDS: {frds_value}")